## Install PySpark and create a session

In [2]:
!pip install pyspark==4.1.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-4.1.2-py2.py3-none-any.whl size=456079457 sha256=4a94a6a2083f1e59eee5f6743de32d4c8c5369a225ef03bfe2c3f30eea37ea36
  Stored in directory: /root/.cache/pip/wheels/e6/9c/35/b08622081a09dc48b9467b570ae170519430915aa3c8d27cf9
Successfully built pyspark
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.3
    Uninstalling pyspark-4.0.3:
      Successfully uninstalled pyspark-4.0.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 4.1.2 which is incompatible.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import DataFrameReader

#create a spark session
spark=SparkSession.builder.appName("AirQuality").getOrCreate()
spark

## Read in data to pyspark dataframe

In [4]:
#Mount Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
#define folder path
path='/content/drive/MyDrive/AirQualityModel/SiteData'

#import datetime
from datetime import datetime,date,time

#Read a single CSV to check inferred schema
df=spark.read.csv('/content/drive/MyDrive/AirQualityModel/SiteData/AQ_Site1.csv',header=True,inferSchema=True)

In [6]:
#Check the top 5 rows to see that data has read in as expected
df.show(5)

+----------+--------+------------------------------------------+----------------+-----------------------+-------+-------------------+-------+--------------------+---------+---------+----------------+---------------+-------+
|      Date|    Time|PM2.5 particulate matter (Hourly measured)|         Status3|Modelled Wind Direction|Status5|Modelled Wind Speed|Status7|           Site Name| Latitude|Longitude|       Site Type|Local Authority|   Zone|
+----------+--------+------------------------------------------+----------------+-----------------------+-------+-------------------+-------+--------------------+---------+---------+----------------+---------------+-------+
|01/01/2025|01:00:00|                                     7.453|V ugm-3 (Ref.eq)|                  231.4|  N deg|               10.9| N ms-1|Borehamwood Meado...|51.661229| -0.27055|Urban Background|      Hertsmere|Eastern|
|01/01/2025|02:00:00|                                     4.151|V ugm-3 (Ref.eq)|                  231.9

In [7]:
#Check the schema
df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- PM2.5 particulate matter (Hourly measured): string (nullable = true)
 |-- Status3: string (nullable = true)
 |-- Modelled Wind Direction: string (nullable = true)
 |-- Status5: string (nullable = true)
 |-- Modelled Wind Speed: string (nullable = true)
 |-- Status7: string (nullable = true)
 |-- Site Name: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Site Type: string (nullable = true)
 |-- Local Authority: string (nullable = true)
 |-- Zone: string (nullable = true)



#The inferred schema has mis-defined some fields as strings, when they should be dates, times floats and doubles

Define a new schema that better matches the expected data

In [8]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType,DoubleType,TimestampType,DateType,TimeType

#define the schema
schema = StructType([
    StructField("Date", DateType(), nullable=True),
    StructField("Time", TimestampType(), nullable=True),
    StructField("Particulate reading", DoubleType(), nullable=True),
    StructField("Particulate Status", StringType(),nullable=True),
    StructField("Wind Direction",FloatType(), nullable=True),
    StructField("Wind Direction Status",StringType(),nullable=True),
    StructField("Wind Speed",FloatType(),nullable=True),
    StructField("wind Speed Status",StringType(),nullable=True),
    StructField("Site Name",StringType(),nullable=True),
    StructField("Latitude",DoubleType(),nullable=True),
    StructField("Longitude",DoubleType(),nullable=True),
    StructField("SiteType",StringType(),nullable=True),
    StructField("Local Authority",StringType(),nullable=True),
    StructField("Zone",StringType(),nullable=True)
    ])

#Read a single CSV to confirm that new schema has been applied
df2=spark.read.csv('/content/drive/MyDrive/AirQualityModel/SiteData/AQ_Site1.csv',header=True,schema=schema)
df2.printSchema()



root
 |-- Date: date (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Particulate reading: double (nullable = true)
 |-- Particulate Status: string (nullable = true)
 |-- Wind Direction: float (nullable = true)
 |-- Wind Direction Status: string (nullable = true)
 |-- Wind Speed: float (nullable = true)
 |-- wind Speed Status: string (nullable = true)
 |-- Site Name: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- SiteType: string (nullable = true)
 |-- Local Authority: string (nullable = true)
 |-- Zone: string (nullable = true)



In [9]:
df2.show(5)

+----+-------------------+-------------------+------------------+--------------+---------------------+----------+-----------------+--------------------+---------+---------+----------------+---------------+-------+
|Date|               Time|Particulate reading|Particulate Status|Wind Direction|Wind Direction Status|Wind Speed|wind Speed Status|           Site Name| Latitude|Longitude|        SiteType|Local Authority|   Zone|
+----+-------------------+-------------------+------------------+--------------+---------------------+----------+-----------------+--------------------+---------+---------+----------------+---------------+-------+
|NULL|2026-07-16 01:00:00|              7.453|  V ugm-3 (Ref.eq)|         231.4|                N deg|      10.9|           N ms-1|Borehamwood Meado...|51.661229| -0.27055|Urban Background|      Hertsmere|Eastern|
|NULL|2026-07-16 02:00:00|              4.151|  V ugm-3 (Ref.eq)|         231.9|                N deg|      11.3|           N ms-1|Borehamwood M

# Having confirmed the new schema is being applied on a single CSV, we now read all the CSVs together into a single dataframe

However, we can see that the data has returned as NULL, as it is not in the default format - when loading in the full dataset, we will use the dataFormat option to define the date format used

In [10]:
#read all csv files in folder to single dataframe
df=(spark.read
    .format('csv')
    .option('header',True)
    .option('dateFormat','dd/MM/yyyy')
    .option('timeFormat',"HH:mm:ss")
    .schema(schema)
    .load(path))
#Check the schema
df.printSchema()


root
 |-- Date: date (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Particulate reading: double (nullable = true)
 |-- Particulate Status: string (nullable = true)
 |-- Wind Direction: float (nullable = true)
 |-- Wind Direction Status: string (nullable = true)
 |-- Wind Speed: float (nullable = true)
 |-- wind Speed Status: string (nullable = true)
 |-- Site Name: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- SiteType: string (nullable = true)
 |-- Local Authority: string (nullable = true)
 |-- Zone: string (nullable = true)



Defining the time field as a timestamp has caused an arbitary date to be added; As the data is hourly measurements, we will use the hour() function to add a column containing an integer representing the hour

In [11]:
#add required functions
import pyspark.sql.functions as sf
from pyspark.sql import Column
#Add "hour" column
df=df.withColumn('Hour',sf.hour(df.Time))
#re-cast Time to string format by removing the date element
df=df.withColumn('Time',sf.date_format(sf.col("Time"), "HH:mm:ss"))
#Check output
df.show(5)
df.printSchema()

+----------+--------+-------------------+------------------+--------------+---------------------+----------+-----------------+--------------------+---------+---------+----------------+---------------+-------+----+
|      Date|    Time|Particulate reading|Particulate Status|Wind Direction|Wind Direction Status|Wind Speed|wind Speed Status|           Site Name| Latitude|Longitude|        SiteType|Local Authority|   Zone|Hour|
+----------+--------+-------------------+------------------+--------------+---------------------+----------+-----------------+--------------------+---------+---------+----------------+---------------+-------+----+
|2025-01-01|01:00:00|             12.618|  V ugm-3 (Ref.eq)|         236.5|                N deg|      13.7|           N ms-1|Peterborough Gart...|52.594012|-0.248945|Urban Background|   Peterborough|Eastern|   1|
|2025-01-01|02:00:00|              3.325|  V ugm-3 (Ref.eq)|         236.0|                N deg|      14.2|           N ms-1|Peterborough Gart.

# Initial Data overview and exploration

In [12]:
#Start with using the describe function on the core columns (non-status) to get a picture of the data
df.select("Date","Particulate reading","Wind Direction","Wind Speed","Hour").describe().show()

+-------+-------------------+-----------------+------------------+-----------------+
|summary|Particulate reading|   Wind Direction|        Wind Speed|             Hour|
+-------+-------------------+-----------------+------------------+-----------------+
|  count|              94254|            94512|             94512|           100740|
|   mean|  8.635067296878514| 190.751974356321| 4.924885728524972|             12.0|
| stddev|  7.937105028580018|96.91355478251096|2.5305540503679733|6.633282503576391|
|    min|               -4.0|              0.0|               0.0|                1|
|    max|            179.647|            360.0|              21.7|               23|
+-------+-------------------+-----------------+------------------+-----------------+



There is a spread of values across the target variable (particulate reading); these will be plotted to look in more detail.

We will also examine the data for indications of correlation between the target variable and the others

In [13]:
#We also need to examine the categorical fields, specifically site type
df.groupBy(df.SiteType).count().show()

+----------------+-----+
|        SiteType|count|
+----------------+-----+
|            NULL|   48|
|   Urban Traffic|26280|
|Rural Background|35040|
|Urban Background|43800|
+----------------+-----+



We can see that there are 3 main site types; reasonaly distributed across all three, but with a slight skew towards 'Urban background'.  There is a small amount of null values; given the small number, these rows can be removed from the dataset

Comparing the total count by SiteType (105,168) and the counts by numerical fields above, we can see that there is a reasonable amount of missing data (~10%).

#Create additional fields to add context


# Create test and train datasets, ensuring that splits are suitably stratified

# Explore training dataset

# Explore models for predicting air quality levels

# Select the best model and use cross-validation to tune the hyperparameters

# Create pipeline to prepare data and run selected model on it

#Run pipeline on Test data